# Sandbox — Silver Orders

Exploração por trás de `silver_rules.tratar_orders`. Lemos a Bronze de `orders` e desenhamos o tratamento: normalizar `order_status` e **converter as 5 colunas de data** (que na Bronze são string) para timestamp — essencial para calcular prazos de entrega na Gold.

In [ ]:
from utils.spark_utils import create_spark_session
from pyspark.sql.functions import col, trim, lower, to_timestamp, current_timestamp

spark = create_spark_session('Sandbox-Silver-Orders')

df_bronze = spark.read.parquet('s3a://bronze/olist/orders')
df_bronze.show(5, truncate=False)
df_bronze.printSchema()

As colunas de data estão como **string**. Vamos conferir os valores distintos de `order_status` (para decidir a normalização) e o formato das datas.

In [ ]:
df_bronze.select('order_status').distinct().show()
df_bronze.select('order_purchase_timestamp').show(5, truncate=False)

`order_status` tem valores como `delivered`, `shipped`, etc. — padronizamos em minúsculo. As datas estão no formato `yyyy-MM-dd HH:mm:ss`, que o `to_timestamp` parseia sem precisar de formato explícito. Colunas de data podem ter nulos (pedido não entregue), e isso é esperado — o cast preserva o null.

In [ ]:
df_silver = (df_bronze
    .withColumn('order_id', trim(col('order_id')))
    .withColumn('customer_id', trim(col('customer_id')))
    .withColumn('order_status', lower(trim(col('order_status'))))
    .withColumn('order_purchase_timestamp', to_timestamp(col('order_purchase_timestamp')))
    .withColumn('order_approved_at', to_timestamp(col('order_approved_at')))
    .withColumn('order_delivered_carrier_date', to_timestamp(col('order_delivered_carrier_date')))
    .withColumn('order_delivered_customer_date', to_timestamp(col('order_delivered_customer_date')))
    .withColumn('order_estimated_delivery_date', to_timestamp(col('order_estimated_delivery_date')))
    .withColumn('dt_criacao_silver', current_timestamp()))

df_silver.printSchema()
df_silver.show(5, truncate=False)

As datas agora são `timestamp` de verdade. **Esta lógica foi promovida para `silver_rules.tratar_orders`.**

In [ ]:
spark.stop()